# E-Commerce Data Cleaning & SQL Analysis

Kenyan online grocery shop, full year 2023.

**Steps:** load the raw CSV, inspect it, clean it, load it into a SQLite database, then answer the business questions with SQL (queries live in `sql/analysis_queries.sql`).

**Business questions**
1. Is the business healthy, and which way is it trending?
2. What drives value: products, locations, customer segments?
3. Who are the best customers, and who is at risk?
4. Where are orders being lost to cancellations?

In [4]:
import os
import sqlite3
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)


def find(*candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"None of these paths exist: {candidates}")


RAW_PATH = find("../kenyan_online_shop_data.csv", "kenyan_online_shop_data.csv")
SQL_PATH = find("../sql/analysis_queries.sql", "sql/analysis_queries.sql", "analysis_queries.sql")
DB_PATH = "kenya_shop.db"
TABLE = "kenya_shop_online_data"

## 1. Load and inspect the raw data

In [5]:
raw = pd.read_csv(RAW_PATH)
print("Shape:", raw.shape)
raw.head()

Shape: (1500, 9)


,InvoiceNo,CustomerID,Product,Quantity,UnitPrice,TotalPrice,InvoiceDate,Location,CustomerType
0,ORD10000,CUST1076,Sanitary Towels (Pack),1,153.60,153.60,2/17/2023,Umoja,Regular
1,ORD10001,CUST1075,Bread (400g),1,52.55,52.55,2/9/2023,Nairobi,Regular
2,ORD10002,CUST1020,Milk (500ml),1,44.20,44.20,2/1/2023,Roysambu,Regular
3,ORD10003,CUST1181,Sanitary Towels (Pack),1,250.33,250.33,2/22/2023,Ruiru,Budget
4,ORD10004,CUST1079,Sugar (1kg),1,132.24,132.24,4/13/2023,Nairobi,Budget


In [6]:
print(raw.dtypes, "\n")
print("Missing values per column:")
print(raw.isna().sum(), "\n")
print("Exact duplicate rows:", raw.duplicated().sum())
print("Unique invoices:", raw["InvoiceNo"].nunique())
print("Unique customers:", raw["CustomerID"].nunique())
print("Unique products:", raw["Product"].nunique())
print("Unique locations:", raw["Location"].nunique())
raw.describe(include="all").T

InvoiceNo           str
CustomerID          str
Product             str
Quantity          int64
UnitPrice       float64
TotalPrice      float64
InvoiceDate         str
Location            str
CustomerType        str
dtype: object 

Missing values per column:
InvoiceNo       0
CustomerID      0
Product         0
Quantity        0
UnitPrice       0
TotalPrice      0
InvoiceDate     0
Location        0
CustomerType    0
dtype: int64 

Exact duplicate rows: 0
Unique invoices: 1500
Unique customers: 299
Unique products: 10
Unique locations: 8


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,1500,1500,ORD10000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CustomerID,1500,299,CUST1020,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,1500,10,Eggs (Tray),182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,"1,500.00",NaN,NaN,NaN,1.00,0.49,0.00,1.00,1.00,1.00,5.00
UnitPrice,"1,500.00",NaN,NaN,NaN,147.60,97.61,29.01,59.59,124.37,215.28,366.89
TotalPrice,"1,500.00",NaN,NaN,NaN,147.19,116.56,0.00,57.09,121.44,217.40,"1,152.40"
InvoiceDate,1500,271,1/31/2023,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,1500,8,Nairobi,485,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CustomerType,1500,3,Budget,921,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Clean the data
Each step prints how many rows it affects, so the cleaning is easy to audit. Orders with a quantity of 0 are cancellations, so they are flagged in an `OrderStatus` column instead of being deleted.

In [ ]:
df = raw.copy()


n = len(df)
df = df.drop_duplicates()
print("Duplicate rows removed:", n - len(df))


key_cols = ["InvoiceNo", "InvoiceDate", "CustomerID", "Product",
            "Quantity", "UnitPrice", "TotalPrice"]
n = len(df)
df = df.dropna(subset=key_cols)
print("Rows with missing key fields removed:", n - len(df))


for col in ["InvoiceNo", "CustomerID", "Product", "Location", "CustomerType"]:
    df[col] = df[col].astype(str).str.strip()
for col in ["Location", "CustomerType"]:
    df[col] = df[col].str.title()


df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
n = len(df)
df = df.dropna(subset=["InvoiceDate"])
print("Rows with unreadable dates removed:", n - len(df))


num_cols = ["Quantity", "UnitPrice", "TotalPrice"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
n = len(df)
df = df.dropna(subset=num_cols)
print("Rows with non-numeric values removed:", n - len(df))


cancelled = (df["Quantity"] == 0) & (df["TotalPrice"] == 0)
valid_sale = (df["Quantity"] > 0) & (df["UnitPrice"] > 0) & (df["TotalPrice"] > 0)
invalid = ~(cancelled | valid_sale)
print("Cancelled orders (Quantity = 0, TotalPrice = 0), kept and flagged:", cancelled.sum())
print("Other impossible rows removed:", invalid.sum())
df = df[~invalid].copy()
df["OrderStatus"] = df["Quantity"].eq(0).map({True: "Cancelled", False: "Completed"})

print("\nRows after cleaning:", len(df))

Duplicate rows removed: 0
Rows with missing key fields removed: 0
Rows with unreadable dates removed: 0
Rows with non-numeric values removed: 0
Cancelled orders (Quantity = 0, TotalPrice = 0), kept and flagged: 78
Other impossible rows removed: 0

Rows after cleaning: 1500


### Validation checks

In [ ]:

mismatch = (df["TotalPrice"] - df["Quantity"] * df["UnitPrice"]).abs() > 0.01
print("Rows where TotalPrice != Quantity x UnitPrice:", mismatch.sum())


print("Date range:", df["InvoiceDate"].min().date(), "to", df["InvoiceDate"].max().date())


print("\nCustomer types:", sorted(df["CustomerType"].unique()))
print("Locations:", sorted(df["Location"].unique()))
print("Products:", sorted(df["Product"].unique()))

Rows where TotalPrice != Quantity x UnitPrice: 0
Date range: 2023-01-01 to 2023-12-31

Customer types: ['Budget', 'Premium', 'Regular']
Locations: ['Kitengela', 'Nairobi', 'Rongai', 'Roysambu', 'Ruaka', 'Ruiru', 'Thika', 'Umoja']
Products: ['Bread (400g)', 'Cooking Oil (1L)', 'Eggs (Tray)', 'Maize Flour (2kg)', 'Milk (500ml)', 'Rice (1kg)', 'Sanitary Towels (Pack)', 'Soap (Bar)', 'Sugar (1kg)', 'Tea Leaves (250g)']


## 3. Add date features
`MonthNum` lets reports and queries sort months in calendar order instead of alphabetically.

In [9]:
df["Year"] = df["InvoiceDate"].dt.year
df["Quarter"] = "Q" + df["InvoiceDate"].dt.quarter.astype(str)
df["MonthNum"] = df["InvoiceDate"].dt.month
df["MonthName"] = df["InvoiceDate"].dt.month_name()
df["InvoiceDate"] = df["InvoiceDate"].dt.strftime("%Y-%m-%d")

df = df[["InvoiceNo", "InvoiceDate", "Year", "Quarter", "MonthNum", "MonthName",
         "CustomerID", "CustomerType", "Location", "Product",
         "Quantity", "UnitPrice", "TotalPrice", "OrderStatus"]].reset_index(drop=True)
df.head()

,InvoiceNo,InvoiceDate,Year,Quarter,MonthNum,MonthName,CustomerID,CustomerType,Location,Product,Quantity,UnitPrice,TotalPrice,OrderStatus
0,ORD10000,2023-02-17,2023,Q1,2,February,CUST1076,Regular,Umoja,Sanitary Towels (Pack),1,153.60,153.60,Completed
1,ORD10001,2023-02-09,2023,Q1,2,February,CUST1075,Regular,Nairobi,Bread (400g),1,52.55,52.55,Completed
2,ORD10002,2023-02-01,2023,Q1,2,February,CUST1020,Regular,Roysambu,Milk (500ml),1,44.20,44.20,Completed
3,ORD10003,2023-02-22,2023,Q1,2,February,CUST1181,Budget,Ruiru,Sanitary Towels (Pack),1,250.33,250.33,Completed
4,ORD10004,2023-04-13,2023,Q2,4,April,CUST1079,Budget,Nairobi,Sugar (1kg),1,132.24,132.24,Completed


### Reconcile with the cleaned dataset used in the Power BI report
The row count and total revenue here should match the report's numbers.

In [10]:
print("Rows in this notebook:", len(df))
print("Total revenue in this notebook:", round(df["TotalPrice"].sum(), 2))

ref_path = next((p for p in ["../cleaned_kenyan_ecom_dataset.csv",
                             "cleaned_kenyan_ecom_dataset.csv"] if os.path.exists(p)), None)
if ref_path:
    ref = pd.read_csv(ref_path)
    print("\nRows in cleaned_kenyan_ecom_dataset.csv:", len(ref))
    print("Total revenue in cleaned_kenyan_ecom_dataset.csv:", round(ref["TotalPrice"].sum(), 2))
else:
    print("\n(cleaned_kenyan_ecom_dataset.csv not found, skipping comparison)")

Rows in this notebook: 1500
Total revenue in this notebook: 220783.49

(cleaned_kenyan_ecom_dataset.csv not found, skipping comparison)


## 4. Load into SQLite

In [11]:
with sqlite3.connect(DB_PATH) as conn:
    df.to_sql(TABLE, conn, if_exists="replace", index=False)
    print(pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {TABLE}", conn))

   row_count
0       1500


## 5. Answer the business questions with SQL
The queries are stored in `sql/analysis_queries.sql`. This helper reads that file and runs a query by name.

In [12]:
def load_queries(path):
    queries, name, buf = {}, None, []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("-- name:"):
                if name:
                    queries[name] = "".join(buf).strip()
                name, buf = line.split(":", 1)[1].strip(), []
            elif name:
                buf.append(line)
    if name:
        queries[name] = "".join(buf).strip()
    return queries


queries = load_queries(SQL_PATH)


def run(name):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql(queries[name], conn)


print("Available queries:", list(queries))

Available queries: ['kpis', 'monthly_trend', 'first_four_months_vs_rest', 'revenue_by_product', 'revenue_by_location', 'segment_summary', 'top_customers', 'at_risk_customers', 'cancellation_by_product', 'cancellation_by_location', 'cancellation_by_customer_type', 'cancellation_by_month']


### 5.1 Is the business healthy? Headline numbers

In [13]:
run("kpis")

,total_revenue,total_orders,completed_orders,cancelled_orders,cancellation_rate_pct,unique_customers,avg_order_value
0,"220,783.49",1500,1422,78,5.20,299,155.26


### 5.2 Which way is it trending? Monthly revenue and orders

In [14]:
run("monthly_trend")

,MonthNum,MonthName,orders,revenue,pct_of_year
0,1,January,387,"63,655.31",28.80
1,2,February,277,"38,075.07",17.20
2,3,March,314,"45,642.41",20.70
3,4,April,283,"39,177.54",17.70
4,5,May,30,"4,215.10",1.90
5,6,June,29,"4,125.80",1.90
6,7,July,26,"3,970.21",1.80
7,8,August,26,"4,733.36",2.10
8,9,September,35,"4,285.28",1.90
9,10,October,33,"5,191.33",2.40


### 5.3 How concentrated is the year? January-April vs May-December

In [15]:
run("first_four_months_vs_rest")

,period,orders,active_customers,revenue,pct_of_revenue
0,Jan-Apr,1261,299,"186,550.33",84.50
1,May-Dec,239,156,"34,233.16",15.50


### 5.4 What drives value? Revenue by product

In [16]:
run("revenue_by_product")

,Product,orders,units_sold,revenue,pct_of_revenue
0,Eggs (Tray),182,176,"58,795.37",26.60
1,Cooking Oil (1L),152,150,"38,231.03",17.30
2,Sanitary Towels (Pack),173,183,"36,482.87",16.50
3,Maize Flour (2kg),146,150,"23,060.54",10.40
4,Rice (1kg),156,152,"18,074.46",8.20
5,Sugar (1kg),133,135,"15,659.26",7.10
6,Tea Leaves (250g),136,136,"9,008.18",4.10
7,Bread (400g),129,131,"8,020.51",3.60
8,Milk (500ml),145,142,"7,245.21",3.30
9,Soap (Bar),148,152,"6,206.06",2.80


### 5.5 Which locations matter most?

In [17]:
run("revenue_by_location")

,Location,customers,orders,revenue,pct_of_revenue
0,Nairobi,96,485,"71,526.60",32.40
1,Thika,44,215,"31,476.26",14.30
2,Ruiru,34,179,"28,075.80",12.70
3,Kitengela,36,186,"27,567.13",12.50
4,Roysambu,31,162,"24,076.72",10.90
5,Rongai,25,141,"19,087.94",8.60
6,Ruaka,19,77,"12,373.65",5.60
7,Umoja,14,55,"6,599.39",3.00


### 5.6 Which customer segments carry the revenue?

In [18]:
run("segment_summary")

,CustomerType,customers,pct_of_customers,revenue,pct_of_revenue,revenue_per_customer
0,Budget,176,58.90,"134,923.67",61.10,766.61
1,Regular,94,31.40,"60,669.86",27.50,645.42
2,Premium,29,9.70,"25,189.96",11.40,868.62


### 5.7 Who are the best customers?

In [19]:
run("top_customers")

,CustomerID,CustomerType,Location,completed_orders,revenue,last_order
0,CUST1043,Premium,Nairobi,6,"2,188.71",2023-01-29
1,CUST1241,Premium,Kitengela,6,"2,113.47",2023-01-29
2,CUST1248,Premium,Ruiru,5,"2,088.60",2023-01-31
3,CUST1240,Premium,Nairobi,9,"1,935.18",2023-01-31
4,CUST1155,Budget,Nairobi,11,"1,919.78",2023-03-22
5,CUST1158,Budget,Nairobi,10,"1,844.20",2023-10-06
6,CUST1154,Premium,Thika,6,"1,833.06",2023-01-22
7,CUST1049,Budget,Nairobi,8,"1,827.04",2023-09-16
8,CUST1060,Budget,Ruaka,7,"1,823.91",2023-10-29
9,CUST1197,Regular,Thika,8,"1,690.88",2023-04-20


### 5.8 Who is at risk? Above-average spenders with no order in the last 90 days

In [20]:
run("at_risk_customers")

,CustomerID,CustomerType,orders,revenue,last_order,days_since_last_order
0,CUST1043,Premium,6,"2,188.71",2023-01-29,336
1,CUST1241,Premium,6,"2,113.47",2023-01-29,336
2,CUST1248,Premium,5,"2,088.60",2023-01-31,334
3,CUST1240,Premium,9,"1,935.18",2023-01-31,334
4,CUST1155,Budget,11,"1,919.78",2023-03-22,284
5,CUST1154,Premium,6,"1,833.06",2023-01-22,343
6,CUST1049,Budget,8,"1,827.04",2023-09-16,106
7,CUST1197,Regular,8,"1,690.88",2023-04-20,255
8,CUST1227,Budget,9,"1,595.93",2023-07-22,162
9,CUST1050,Premium,4,"1,589.31",2023-01-22,343


### 5.9 Where are orders being lost? Overall and by product

In [21]:
run("cancellation_by_product")

,Product,orders,cancelled,cancellation_rate_pct
0,Milk (500ml),145,12,8.30
1,Bread (400g),129,10,7.80
2,Sanitary Towels (Pack),173,10,5.80
3,Sugar (1kg),133,7,5.30
4,Tea Leaves (250g),136,7,5.10
5,Soap (Bar),148,7,4.70
6,Rice (1kg),156,7,4.50
7,Maize Flour (2kg),146,6,4.10
8,Cooking Oil (1L),152,6,3.90
9,Eggs (Tray),182,6,3.30


### 5.10 Cancellations by location

In [22]:
run("cancellation_by_location")

,Location,orders,cancelled,cancellation_rate_pct
0,Ruaka,77,6,7.80
1,Ruiru,179,13,7.30
2,Roysambu,162,11,6.80
3,Rongai,141,8,5.70
4,Umoja,55,3,5.50
5,Nairobi,485,24,4.90
6,Thika,215,8,3.70
7,Kitengela,186,5,2.70


### 5.11 Cancellations by customer type

In [23]:
run("cancellation_by_customer_type")

,CustomerType,orders,cancelled,cancellation_rate_pct
0,Premium,128,8,6.30
1,Budget,921,48,5.20
2,Regular,451,22,4.90


### 5.12 Cancellations by month

In [24]:
run("cancellation_by_month")

,MonthNum,MonthName,orders,cancelled,cancellation_rate_pct
0,1,January,387,20,5.20
1,2,February,277,14,5.10
2,3,March,314,21,6.70
3,4,April,283,11,3.90
4,5,May,30,1,3.30
5,6,June,29,2,6.90
6,7,July,26,2,7.70
7,8,August,26,0,0.00
8,9,September,35,0,0.00
9,10,October,33,1,3.00
